###Ex. No: 7
Fine-tune pre-trained language models for domainspecific applications

In [ ]:
!pip install transformers datasets evaluate accelerate torch -q

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline
)

# 1. Domain-specific dataset
data = {
    "text": [
        "The transformer model achieved excellent accuracy.",
        "Large Language Models are revolutionizing AI.",
        "The football team won the championship.",
        "The cricket match was exciting.",
        "Neural networks are widely used in deep learning.",
        "The player scored a brilliant goal.",
        "Machine learning improves decision making.",
        "The tennis tournament starts tomorrow."
    ],
    "label": [1, 1, 0, 0, 1, 0, 1, 0]
}
dataset = Dataset.from_dict(data)

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3. Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# 4. Training
training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    per_device_train_batch_size=2,
    num_train_epochs=3,          # increased a bit for better results
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
trainer.train()

# 5. Save
trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

# 6. Inference
classifier = pipeline("text-classification", model="./fine_tuned_model", tokenizer="./fine_tuned_model")

text = "Generative AI models improve intelligent automation."
result = classifier(text)

labels = {"LABEL_0": "Sports", "LABEL_1": "Technology"}

print("\nPrediction")
print("-------------------------")
print("Input :", text)
print("Predicted Class :", labels[result[0]["label"]])
print("Confidence Score :", round(result[0]["score"], 3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)